In [ ]:
import pandas as pd
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog

In [ ]:


# Ocultar la ventana principal de tkinter
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# Paso 1: Seleccionar el archivo de ventas
nombre_archivo_ventas = filedialog.askopenfilename(
    title="Selecciona el archivo de ventas (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_ventas:
    print("No se seleccionó el archivo de ventas.")
    exit()

# Paso 2: Cargar el archivo de ventas
try:
    df_ventas = pd.read_excel(nombre_archivo_ventas)
    print(f"Archivo de ventas '{os.path.basename(nombre_archivo_ventas)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de ventas: {e}")
    exit()

# Paso 3: Seleccionar el archivo de notas crédito
nombre_archivo_notas_credito = filedialog.askopenfilename(
    title="Selecciona el archivo de notas crédito (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_notas_credito:
    print("No se seleccionó el archivo de notas crédito.")
    exit()

# Paso 4: Cargar el archivo de notas crédito
try:
    df_notas_credito = pd.read_excel(nombre_archivo_notas_credito)
    print(f"Archivo de notas crédito '{os.path.basename(nombre_archivo_notas_credito)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de notas crédito: {e}")
    exit()


In [ ]:

# Paso 5: Extraer el número de factura de la columna "Referencia" en las notas crédito
df_notas_credito['NUMERO_FACTURA'] = df_notas_credito['Líneas de factura/Referencia'].apply(
    lambda x: re.search(r'(FEVY\d+)', x).group(1) if pd.notna(x) and re.search(r'(FEVY\d+)', x) else None
)
df_notas_credito = df_notas_credito.drop(columns=['Líneas de factura/Número'])
df_notas_credito = df_notas_credito.rename(columns={'NUMERO_FACTURA': 'Líneas de factura/Número'})
# Paso 6: Convertir las cantidades y totales de las notas crédito a valores negativos
df_notas_credito['Líneas de factura/Cantidad'] = -df_notas_credito['Líneas de factura/Cantidad']
df_notas_credito['Líneas de factura/Total'] = -df_notas_credito['Líneas de factura/Total']


In [ ]:
# Paso 8: Crear una columna temporal que combine NUMERO_FACTURA y PRODUCTO
df_ventas['NUMERO_FACTURA-PRODUCTO'] = df_ventas['Líneas de factura/Número'] + '-' + df_ventas['Líneas de factura/Producto']
df_notas_credito['NUMERO_FACTURA-PRODUCTO'] = df_notas_credito['Líneas de factura/Número'] + '-' + df_notas_credito['Líneas de factura/Producto']
# Paso 9: Filtrar las notas crédito para incluir solo las que coinciden con ventas existentes
notas_credito_validas = df_notas_credito['NUMERO_FACTURA-PRODUCTO'].isin(df_ventas['NUMERO_FACTURA-PRODUCTO'])
df_notas_credito_filtrado = df_notas_credito[notas_credito_validas]
# Paso 8: Combinar ambos datasets (ventas y notas crédito)
df_consolidado = pd.concat([df_ventas, df_notas_credito_filtrado], ignore_index=True)
# Paso 10: Agrupar por la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado = df_consolidado.groupby(
    'NUMERO_FACTURA-PRODUCTO',  # Agrupar por la combinación de factura y producto
    as_index=False
).agg({
    'Líneas de factura/Fecha de factura': 'first',
    'Líneas de factura/Asociado': 'first',
    'Líneas de factura/Número': 'first',
    'Líneas de factura/Producto': 'first',
    'Líneas de factura/Cantidad': 'sum',  # Sumar las cantidades
    'Líneas de factura/Total': 'sum',     # Sumar los totales
    'Líneas de factura/Moneda/Tasa actual': 'first',
    'Líneas de factura/Asociado/Número de Identificación': 'first',
    'Líneas de factura/Asociado/Teléfono': 'first',
    'Líneas de factura/Asociado/Correo electrónico': 'first',
    'Líneas de factura/Asociado/Ciudad': 'first',
    'Líneas de factura/Asociado/Estado': 'first',
    'Equipo de Ventas': 'first',
    'Líneas de factura/Referencia': 'first',
    'Asesor Comercial': 'first',
    'Origen': 'first',
    'Origen/Nombre de la Fuente': 'first',
    'Tipo de cliente': 'first',
    'Etiqueta contacto': 'first'

})
# Paso 12: Eliminar la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado.drop(columns=['NUMERO_FACTURA-PRODUCTO'], inplace=True)
# Paso 13: Filtrar solo las filas donde la cantidad sea mayor que 0 (eliminar ventas canceladas)
df_consolidado = df_consolidado[df_consolidado['Líneas de factura/Cantidad'] > 0]
# Paso 14: Generar el nombre del archivo de salida






# Nueva logica

In [22]:
df_consolidado.columns

Index(['Líneas de factura/Fecha de factura', 'Líneas de factura/Asociado',
       'Líneas de factura/Número', 'Líneas de factura/Producto',
       'Líneas de factura/Cantidad', 'Líneas de factura/Total',
       'Líneas de factura/Moneda/Tasa actual',
       'Líneas de factura/Asociado/Número de Identificación',
       'Líneas de factura/Asociado/Teléfono',
       'Líneas de factura/Asociado/Correo electrónico',
       'Líneas de factura/Asociado/Ciudad',
       'Líneas de factura/Asociado/Estado', 'Equipo de Ventas',
       'Líneas de factura/Referencia', 'Asesor Comercial', 'Origen',
       'Origen/Nombre de la Fuente', 'Tipo de cliente', 'Etiqueta contacto'],
      dtype='object')

In [ ]:
# Guarda en la variables las ventas sin tipo de cliente y con etiqueta mayorista
etiqueta_mayorista = df_consolidado[(df_consolidado['Tipo de cliente'].isna())&
               (df_consolidado['Etiqueta contacto']=='MAYORISTA NV')
               ] 
# Copia de la etiqueta los clientes mayoristas que aparecen en blanco
df_consolidado.loc[(df_consolidado['Tipo de cliente'].isna())&
               (df_consolidado['Etiqueta contacto']=='MAYORISTA NV'), 'Tipo de cliente'
               ] = 'MAYORISTA NV'

In [30]:
facturas = df_consolidado[df_consolidado['Tipo de cliente'].notna()]

In [35]:
facturas.drop_duplicates(subset='Líneas de factura/Número').reset_index(drop=True)

,Líneas de factura/Fecha de factura,Líneas de factura/Asociado,Líneas de factura/Número,Líneas de factura/Producto,Líneas de factura/Cantidad,Líneas de factura/Total,Líneas de factura/Moneda/Tasa actual,Líneas de factura/Asociado/Número de Identificación,Líneas de factura/Asociado/Teléfono,Líneas de factura/Asociado/Correo electrónico,Líneas de factura/Asociado/Ciudad,Líneas de factura/Asociado/Estado,Equipo de Ventas,Líneas de factura/Referencia,Asesor Comercial,Origen,Origen/Nombre de la Fuente,Tipo de cliente,Etiqueta contacto
0,2025-08-23,ZAR IMPORT ZARIMPORT S.A.,/,[KD01] SHAMPOO KIDS,384.0,1632.0,0.000249,0992534427001,+593 4-232-3511,None,Guayaquil,Guayas (EC),Ventas,None,None,None,None,EXTERIOR,EXTERIOR
1,2025-08-15,CONSUMIDOR FINAL,2YPO1000,[PCN03] DUAL CREMA PARA PEINAR,1.0,28000.0,1.000000,222222222222,+57 301 7249949,administrativo@lapocion.com,CALI,Valle del Cauca (CO),Ventas,Caja 2/2955,None,None,None,CLIENTE,CLIENTE
2,2025-08-15,CONSUMIDOR FINAL,2YPO1001,[PCN09] TERMOPROTECTOR BRILLO INFINITO,1.0,38400.0,1.000000,222222222222,+57 301 7249949,administrativo@lapocion.com,CALI,Valle del Cauca (CO),Ventas,Caja 2/2956,None,None,None,CLIENTE,CLIENTE
3,2025-08-15,CONSUMIDOR FINAL,2YPO1002,[PCN19] DUTONIC (TONICO CAPILAR),1.0,43200.0,1.000000,222222222222,+57 301 7249949,administrativo@lapocion.com,CALI,Valle del Cauca (CO),Ventas,Caja 2/2957,None,None,None,CLIENTE,CLIENTE
4,2025-08-15,CONSUMIDOR FINAL,2YPO1003,[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO,1.0,29600.0,1.000000,222222222222,+57 301 7249949,administrativo@lapocion.com,CALI,Valle del Cauca (CO),Ventas,Caja 2/2959,None,None,None,CLIENTE,CLIENTE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2620,2025-08-05,GARCIA ARANGO GUSTAVO,FVE25706,[ALOJAMIENTO] SERV. ALOJAMIENTO,1.0,1700000.0,1.000000,19086925,+57 315 5519450,garciaavaluos@gmail.com,CALI,Valle del Cauca (CO),Ventas,None,None,None,None,CLIENTE,CLIENTE
2621,2025-08-11,POSSO RODRIGUEZ RAMIRO,FVE25708,[ALOJAMIENTO] SERV. ALOJAMIENTO,1.0,2000000.0,1.000000,16257285,+57 315 6473949,ferroelectricosmundial285@gmail.com,CALI,Valle del Cauca (CO),Ventas,None,None,None,None,CLIENTE,None
2622,2025-08-12,MUELAS RICO LUIS CARLOS,FVE25709,[ALOJAMIENTO] SERV. ALOJAMIENTO,1.0,2300000.0,1.000000,1144070223,None,touchmusicdj@gmail.com,CALI,Valle del Cauca (CO),Ventas,None,None,None,None,CLIENTE,CLIENTE
2623,2025-08-08,ZAR IMPORT ZARIMPORT S.A.,FYEX17,[PCN01] TRATAMIENTO LA POCION,192.0,816.0,0.000249,0992534427001,+593 4-232-3511,None,Guayaquil,Guayas (EC),Ventas,None,None,None,None,EXTERIOR,EXTERIOR


In [ ]:

nombre_archivo_salida = os.path.splitext(nombre_archivo_ventas)[0] + "_procesado.xlsx"

# Paso 15: Guardar el archivo consolidado
try:
    df_consolidado.to_excel(nombre_archivo_salida, index=False)
    print(f"Archivo consolidado guardado como '{nombre_archivo_salida}'.")
except Exception as e:
    print(f"Error al guardar el archivo consolidado: {e}")

# Paso 16: Obtener el listado de facturas afectadas por notas crédito
facturas_afectadas = df_notas_credito_filtrado[['Líneas de factura/Número', 'Líneas de factura/Producto', 'Líneas de factura/Cantidad', 'Líneas de factura/Total']].dropna(subset=['Líneas de factura/Número'])

# Paso 17: Mostrar el listado de facturas afectadas
print("Facturas afectadas por notas crédito:")
print(facturas_afectadas.shape)

# ruta = self.validar_ruta()
# Paso 18: Guardar el listado de facturas afectadas en un archivo Excel (opcional)
nombre_archivo_facturas_afectadas = os.path.splitext(nombre_archivo_ventas)[0] + "_facturas_afectadas.xlsx"
try:
    facturas_afectadas.to_excel(nombre_archivo_facturas_afectadas, index=False)
    print(f"Listado de facturas afectadas guardado como '{nombre_archivo_facturas_afectadas}'.")
except Exception as e:
    print(f"Error al guardar el listado de facturas afectadas: {e}")

# return {'archivo_salida': df_consolidado,
#         'nombre_archivo':nombre_archivo_salida,
#         'facturas_afectadas' :facturas_afectadas}
